# Day 6: File I/O Operations

Today we'll learn how to read from and write to various file formats using pandas.

In [ ]:
import pandas as pd
import numpy as np
import json
from io import StringIO

# Create sample data for demonstrations
sample_data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva'],
    'Age': [25, 30, 35, 28, 32],
    'City': ['New York', 'London', 'Tokyo', 'Paris', 'Berlin'],
    'Salary': [50000, 60000, 70000, 55000, 65000],
    'Department': ['IT', 'Finance', 'IT', 'HR', 'Finance']
}
df = pd.DataFrame(sample_data)
print("Sample DataFrame:")
print(df)

## 1. CSV Files

In [ ]:
# Write to CSV
df.to_csv('sample_data.csv', index=False)
print("Data written to CSV")

# Read from CSV
df_csv = pd.read_csv('sample_data.csv')
print("\nData read from CSV:")
print(df_csv)
print()

# CSV with custom parameters
df.to_csv('custom.csv', sep=';', encoding='utf-8', index=False)
df_custom = pd.read_csv('custom.csv', sep=';', encoding='utf-8')
print("Custom CSV with semicolon separator:")
print(df_custom.head())
print()

# Read specific columns
df_subset = pd.read_csv('sample_data.csv', usecols=['Name', 'Age', 'Salary'])
print("Reading specific columns:")
print(df_subset)
print()

# Read with data type specification
dtypes = {'Name': 'string', 'Age': 'int32', 'Salary': 'float64'}
df_typed = pd.read_csv('sample_data.csv', dtype=dtypes)
print("Data types after reading:")
print(df_typed.dtypes)

## 2. Excel Files

In [ ]:
# Write to Excel
df.to_excel('sample_data.xlsx', index=False, sheet_name='Employees')
print("Data written to Excel")

# Read from Excel
df_excel = pd.read_excel('sample_data.xlsx', sheet_name='Employees')
print("\nData read from Excel:")
print(df_excel)
print()

# Multiple sheets
with pd.ExcelWriter('multi_sheet.xlsx') as writer:
    df[df['Department'] == 'IT'].to_excel(writer, sheet_name='IT', index=False)
    df[df['Department'] == 'Finance'].to_excel(writer, sheet_name='Finance', index=False)
    df[df['Department'] == 'HR'].to_excel(writer, sheet_name='HR', index=False)

print("Multi-sheet Excel created")

# Read all sheets
all_sheets = pd.read_excel('multi_sheet.xlsx', sheet_name=None)
print("\nAll sheets read:")
for sheet_name, sheet_df in all_sheets.items():
    print(f"{sheet_name}: {len(sheet_df)} rows")
print()

# Read specific range
df_range = pd.read_excel('sample_data.xlsx', usecols='A:C', nrows=3)
print("Reading specific range (A:C, first 3 rows):")
print(df_range)

## 3. JSON Files

In [ ]:
# Write to JSON
df.to_json('sample_data.json', orient='records', indent=2)
print("Data written to JSON")

# Read from JSON
df_json = pd.read_json('sample_data.json')
print("\nData read from JSON:")
print(df_json)
print()

# Different JSON orientations
orientations = ['records', 'index', 'values', 'columns']
for orient in orientations:
    filename = f'data_{orient}.json'
    df.to_json(filename, orient=orient, indent=2)
    
    # Read back and show first few characters
    with open(filename, 'r') as f:
        content = f.read()[:100]
    print(f"Orient '{orient}': {content}...")
print()

# Nested JSON
nested_json = '''
[
  {"name": "Alice", "details": {"age": 25, "city": "New York"}},
  {"name": "Bob", "details": {"age": 30, "city": "London"}}
]
'''

df_nested = pd.read_json(StringIO(nested_json))
print("Nested JSON:")
print(df_nested)
print()

# Normalize nested JSON
from pandas import json_normalize
data = json.loads(nested_json)
df_normalized = json_normalize(data)
print("Normalized nested JSON:")
print(df_normalized)

## 4. Parquet Files

In [ ]:
# Write to Parquet (requires pyarrow or fastparquet)
try:
    df.to_parquet('sample_data.parquet', index=False)
    print("Data written to Parquet")
    
    # Read from Parquet
    df_parquet = pd.read_parquet('sample_data.parquet')
    print("\nData read from Parquet:")
    print(df_parquet)
    print()
    
    # Check file sizes
    import os
    csv_size = os.path.getsize('sample_data.csv')
    parquet_size = os.path.getsize('sample_data.parquet')
    print(f"CSV size: {csv_size} bytes")
    print(f"Parquet size: {parquet_size} bytes")
    print(f"Compression ratio: {csv_size/parquet_size:.2f}x")
    
except ImportError:
    print("Parquet support requires pyarrow or fastparquet")
    print("Install with: pip install pyarrow")

## 5. HDF5 Files

In [ ]:
# Write to HDF5 (requires tables)
try:
    df.to_hdf('sample_data.h5', key='employees', mode='w')
    print("Data written to HDF5")
    
    # Read from HDF5
    df_hdf = pd.read_hdf('sample_data.h5', key='employees')
    print("\nData read from HDF5:")
    print(df_hdf)
    print()
    
    # Multiple datasets in one file
    df.to_hdf('multi_data.h5', key='all_employees', mode='w')
    df[df['Department'] == 'IT'].to_hdf('multi_data.h5', key='it_employees', mode='a')
    
    # List keys in HDF5 file
    with pd.HDFStore('multi_data.h5', mode='r') as store:
        print("Keys in HDF5 file:", list(store.keys()))
        
except ImportError:
    print("HDF5 support requires tables")
    print("Install with: pip install tables")

## 6. SQL Databases

In [ ]:
# SQLite example
import sqlite3

# Create connection
conn = sqlite3.connect('sample_database.db')

# Write to SQL
df.to_sql('employees', conn, if_exists='replace', index=False)
print("Data written to SQLite database")

# Read from SQL
df_sql = pd.read_sql('SELECT * FROM employees', conn)
print("\nData read from SQL:")
print(df_sql)
print()

# SQL queries
high_salary = pd.read_sql('''
    SELECT Name, Salary, Department 
    FROM employees 
    WHERE Salary > 55000 
    ORDER BY Salary DESC
''', conn)
print("High salary employees:")
print(high_salary)
print()

# Aggregation in SQL
dept_stats = pd.read_sql('''
    SELECT Department, 
           COUNT(*) as Count,
           AVG(Salary) as Avg_Salary,
           MAX(Age) as Max_Age
    FROM employees 
    GROUP BY Department
''', conn)
print("Department statistics:")
print(dept_stats)

conn.close()

## 7. Web Data

In [ ]:
# Read HTML tables
try:
    # Example: Reading tables from a webpage
    url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'
    tables = pd.read_html(url)
    print(f"Found {len(tables)} tables on the webpage")
    
    # Display first table info
    if tables:
        gdp_table = tables[0]
        print("\nFirst table shape:", gdp_table.shape)
        print("Columns:", gdp_table.columns.tolist())
        print("\nFirst few rows:")
        print(gdp_table.head())
        
except Exception as e:
    print(f"Error reading HTML: {e}")
    print("This might be due to network issues or missing dependencies")

# Read from URL (CSV)
try:
    # Example with a CSV URL
    csv_url = 'https://raw.githubusercontent.com/datasets/covid-19/master/data/countries-aggregated.csv'
    df_url = pd.read_csv(csv_url, nrows=10)  # Read only first 10 rows
    print("\nData from CSV URL:")
    print(df_url)
    
except Exception as e:
    print(f"Error reading CSV from URL: {e}")

## 8. Clipboard Operations

In [ ]:
# Copy to clipboard
try:
    df.head(3).to_clipboard(index=False)
    print("Data copied to clipboard")
    print("You can now paste it in Excel or any text editor")
    
    # Read from clipboard (uncomment to test)
    # df_clipboard = pd.read_clipboard()
    # print("\nData read from clipboard:")
    # print(df_clipboard)
    
except Exception as e:
    print(f"Clipboard operation failed: {e}")
    print("This might require additional system dependencies")

## 9. Performance Considerations

In [ ]:
# Create larger dataset for performance testing
import time

large_df = pd.DataFrame({
    'id': range(100000),
    'value': np.random.randn(100000),
    'category': np.random.choice(['A', 'B', 'C'], 100000),
    'date': pd.date_range('2020-01-01', periods=100000, freq='1min')
})

print(f"Large dataset shape: {large_df.shape}")
print(f"Memory usage: {large_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print()

# Compare write performance
formats = ['csv', 'parquet', 'hdf']
write_times = {}

for fmt in formats:
    try:
        start_time = time.time()
        
        if fmt == 'csv':
            large_df.to_csv('large_data.csv', index=False)
        elif fmt == 'parquet':
            large_df.to_parquet('large_data.parquet', index=False)
        elif fmt == 'hdf':
            large_df.to_hdf('large_data.h5', key='data', mode='w')
            
        write_times[fmt] = time.time() - start_time
        print(f"{fmt.upper()} write time: {write_times[fmt]:.2f} seconds")
        
    except Exception as e:
        print(f"{fmt.upper()} write failed: {e}")

print()

# Compare read performance
read_times = {}

for fmt in formats:
    try:
        start_time = time.time()
        
        if fmt == 'csv':
            df_read = pd.read_csv('large_data.csv')
        elif fmt == 'parquet':
            df_read = pd.read_parquet('large_data.parquet')
        elif fmt == 'hdf':
            df_read = pd.read_hdf('large_data.h5', key='data')
            
        read_times[fmt] = time.time() - start_time
        print(f"{fmt.upper()} read time: {read_times[fmt]:.2f} seconds")
        
    except Exception as e:
        print(f"{fmt.upper()} read failed: {e}")

## 10. Advanced I/O Options

In [ ]:
# Chunked reading for large files
chunk_size = 1000
chunks = []

try:
    for chunk in pd.read_csv('large_data.csv', chunksize=chunk_size):
        # Process each chunk
        processed_chunk = chunk[chunk['value'] > 0]  # Example processing
        chunks.append(processed_chunk)
        
        if len(chunks) >= 5:  # Process only first 5 chunks for demo
            break
    
    # Combine processed chunks
    result = pd.concat(chunks, ignore_index=True)
    print(f"Processed {len(chunks)} chunks, result shape: {result.shape}")
    
except FileNotFoundError:
    print("Large CSV file not found, skipping chunked reading demo")

# Compression
compression_formats = ['gzip', 'bz2', 'zip']

for comp in compression_formats:
    try:
        # Write compressed
        filename = f'compressed_data.csv.{comp}'
        df.to_csv(filename, compression=comp, index=False)
        
        # Read compressed
        df_comp = pd.read_csv(filename, compression=comp)
        
        # Check file size
        import os
        original_size = os.path.getsize('sample_data.csv')
        compressed_size = os.path.getsize(filename)
        
        print(f"{comp.upper()}: {original_size} -> {compressed_size} bytes "
              f"({compressed_size/original_size:.2f}x)")
        
    except Exception as e:
        print(f"{comp.upper()} compression failed: {e}")

print()

# Custom date parsing
date_data = '''
date,value
2024-01-15,100
2024-01-16,110
2024-01-17,105
'''

# Write sample date data
with open('date_data.csv', 'w') as f:
    f.write(date_data)

# Read with date parsing
df_dates = pd.read_csv('date_data.csv', 
                      parse_dates=['date'],
                      date_parser=pd.to_datetime)

print("Data with parsed dates:")
print(df_dates)
print("Date column type:", df_dates['date'].dtype)

## Practice Exercises

In [ ]:
# Exercise 1: Multi-format data pipeline
# Create sample sales data
sales_data = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=100, freq='D'),
    'product': np.random.choice(['A', 'B', 'C'], 100),
    'sales': np.random.randint(100, 1000, 100),
    'region': np.random.choice(['North', 'South', 'East', 'West'], 100)
})

# Save in multiple formats
sales_data.to_csv('sales.csv', index=False)
sales_data.to_json('sales.json', orient='records', date_format='iso')

try:
    sales_data.to_parquet('sales.parquet', index=False)
    print("Sales data saved in CSV, JSON, and Parquet formats")
except:
    print("Sales data saved in CSV and JSON formats")

# Exercise 2: Data validation after I/O
original_shape = sales_data.shape
original_dtypes = sales_data.dtypes

# Read back and validate
sales_csv = pd.read_csv('sales.csv', parse_dates=['date'])
sales_json = pd.read_json('sales.json')

print(f"\nOriginal shape: {original_shape}")
print(f"CSV shape: {sales_csv.shape}")
print(f"JSON shape: {sales_json.shape}")
print(f"\nData integrity check: {sales_data.equals(sales_csv)}")

# Exercise 3: Efficient large file processing
def process_large_file(filename, chunk_size=1000):
    """Process large CSV file in chunks"""
    total_rows = 0
    sum_sales = 0
    
    try:
        for chunk in pd.read_csv(filename, chunksize=chunk_size):
            total_rows += len(chunk)
            sum_sales += chunk['sales'].sum()
        
        return total_rows, sum_sales
    except FileNotFoundError:
        return 0, 0

rows, total_sales = process_large_file('sales.csv')
print(f"\nProcessed {rows} rows, total sales: {total_sales}")

## Tomorrow's Preview
In Day 7, we'll cover:
- Data Visualization with Pandas
- Integration with Matplotlib and Seaborn
- Interactive Plotting
- Statistical Visualizations